# Testing of ECGDataset for ECG-Hubert. 
This should generalize to the ECG-BigBird and ECG-PatchTST models as well

In [1]:
# add autoreload
%load_ext autoreload
%autoreload 2
import neurokit2 as nk
import numpy as np
import pandas as pd
import wfdb
import os
import sys
import re
import dotenv
from collections import defaultdict
from tqdm import tqdm

import matplotlib.pyplot as plt

from torch.utils.data import DataLoader

from transformers import pipeline
from transformers import AutoModel

from torch import float32

import torch

# presets for preprocessing: ECGHubert, ECGFounder Medxai

In [2]:
from timex.ecg import dataset

INFO:root:Initializing Config class


In [3]:
dotenv.load_dotenv('../.env')
BASE_DIR = os.getenv('ECG_DIR')

In [4]:
PREPLIST = ['savgol', 'resampler', 'notch', 'bandpass', 'detrend', 'peak_trimming']

In [5]:
ecgConfig = dataset.Config
ecgConfig.SAMPLING_RATE = 500

ecgDS = dataset.ECGDataset(data=os.path.join(BASE_DIR, 'wilson-central-terminal-ecg-database-1.0.1'),
                           label_binarizer=None,
                           visualisation=False,
                           augmentations=[],
                           preprocessing=PREPLIST,
                           config=ecgConfig,
                           encode=True,
                           pretrain=False)

First 5 elements of the file_list: ['T:\\laupodteam\\AIOS\\Bram\\data\\ECG\\wilson-central-terminal-ecg-database-1.0.1\\patient001\\seg01.hea', 'T:\\laupodteam\\AIOS\\Bram\\data\\ECG\\wilson-central-terminal-ecg-database-1.0.1\\patient001\\seg02.hea', 'T:\\laupodteam\\AIOS\\Bram\\data\\ECG\\wilson-central-terminal-ecg-database-1.0.1\\patient001\\seg03.hea', 'T:\\laupodteam\\AIOS\\Bram\\data\\ECG\\wilson-central-terminal-ecg-database-1.0.1\\patient001\\seg04.hea', 'T:\\laupodteam\\AIOS\\Bram\\data\\ECG\\wilson-central-terminal-ecg-database-1.0.1\\patient002\\seg01.hea']
No labels available to show unique values


In [ ]:
ecgDL = DataLoader(ecgDS, batch_size=128, shuffle=False, collate_fn=ecgDS.collate)

In [7]:
HubertECG = AutoModel.from_pretrained("Edoardo-BS/hubert-ecg-small", trust_remote_code=True,
                                        torch_dtype=float32, low_cpu_mem_usage=True)

In [ ]:
# use the DataLoader to extract the ECG signals
HubertECG.eval()  # Set the model to evaluation mode

results = []
for batch in tqdm(ecgDL, desc="Processing ECG batches"):
    ecg_data, ecg_filenames = batch
    # process each ECG signal in the batch
    with torch.no_grad():
        for signal, filename in zip(ecg_data, ecg_filenames):
            # Here you can apply the Hubert model to the signal
            # For example, you can use the model to extract features or perform classification
            features = HubertECG(signal[:12,:], 
                                attention_mask=None, 
                                output_attentions=False,
                                output_hidden_states=True, 
                                return_dict=True)  # Add batch dimension if needed
            
            channel_embeddings = []
            channel_embeddings_projections = []
            projected_features = HubertECG.final_proj[0](features['last_hidden_state']) 
            for i in range(12):
                # to numpy
                channel_embeddings.append(features['last_hidden_state'][i].mean(dim=0).cpu().numpy())
                channel_embeddings_projections.append(projected_features[i].mean(dim=0).cpu().numpy())

            results.append({
                'filename': filename,
                'channel_embs': channel_embeddings,
                'channel_embs_proj': channel_embeddings_projections
            })

Processing ECG batches:   0%|          | 0/17 [00:00<?, ?it/s]\\DS.UMCUTRECHT.NL\DATA\LAB\laupodteam\AIOS\Bram\production\TimEx\src\timex\ecg\preprocessor.py:59: UserWarning: Number of channels 37 is not as expected: 12
  warnings.warn(f"Number of channels {self.p_signal.shape[0]} is not as expected: {num_channels}")
Processing ECG batches:  24%|██▎       | 4/17 [00:39<02:08,  9.89s/it]

In [ ]:
len(results)  # Check how many signals were processed

540

In [ ]:
# make a dataframe per channel


# create 2d UMAP embeddings per channel

# visualize

array([-4.14191514e-01, -1.05674422e+00,  4.92807269e-01, -9.60537076e-01,
        1.29896641e+00,  1.35920763e+00,  1.61630034e+00, -1.48029399e+00,
        2.62493566e-02,  1.98470250e-01,  2.10533783e-01, -4.81446058e-01,
        9.24287856e-01,  9.26669359e-01, -2.01138902e+00, -1.15482819e+00,
       -5.70778847e-02, -8.25817809e-02, -1.98923957e+00, -1.17112589e+00,
       -1.96346328e-01,  5.41248739e-01, -1.00520873e+00,  2.33219147e-01,
       -8.38366807e-01,  9.39397216e-01, -2.31040254e-01,  1.07748806e+00,
        1.92351425e+00,  9.21024263e-01, -1.05973232e+00,  3.55016440e-01,
        1.27437964e-01,  5.47864020e-01, -1.39831543e-01,  5.40496893e-02,
        5.70171237e-01,  1.57470912e-01, -1.83258903e+00,  5.40401161e-01,
       -1.08559704e+00,  1.54983020e+00, -3.36512238e-01,  3.94038945e-01,
       -6.69730961e-01,  1.44930398e+00,  2.27408862e+00,  1.66772461e+00,
       -1.11749005e+00, -9.65719521e-01, -1.21047042e-01, -1.25075507e+00,
        7.50298023e-01, -